In [ ]:
# paths are relative to this notebook (it lives in the notebooks/ folder of the repo)
TRAVEL_DIR = "../data/processed/travel"                        # Amelia's stop_distance_ft.csv and location_stop_crosswalk.csv
PICKS_FILE = "../data/raw/picks_sep2025_sep2026.csv"           # year of picks (C&D data, not in git)
TRAIN_MONTH = "2026-07"      # month we learn from
TEST_MONTH = "2026-08"       # month we pretend the swaps were already done
DOCK = "D_183183.9_5933.3"   # where every tour starts and ends
GOLDEN_FT = 60               # golden zone = within this many feet of the front of an aisle
MAX_SWAPS = 100
COST_PER_MOVE = 0.17         # dollars; a swap is two moves
DOLLARS_PER_FOOT = 45.50 / 1.25 / 5280    # $45.50 an hour, walking 1.25 mph

In [ ]:
import pandas as pd

# walking distance in feet between any two stops on Amelia's map
distances = pd.read_csv(TRAVEL_DIR + "/stop_distance_ft.csv", index_col=0)
table = distances.values                  # same numbers as a plain grid (fast to look up)
row_number = {stop: i for i, stop in enumerate(distances.index)}
def feet(stop_a, stop_b):
    return table[row_number[stop_a], row_number[stop_b]]

# which stop and aisle each location is at (case-pick faces on the map only)
crosswalk = pd.read_csv(TRAVEL_DIR + "/location_stop_crosswalk.csv", dtype=str)
crosswalk = crosswalk[(crosswalk["status"] == "ok") & (crosswalk["zone"] == "WZPCK") & (crosswalk["slot_type"] == "F")]
stop_of = dict(zip(crosswalk["location"], crosswalk["stop_id"]))
aisle_of = dict(zip(crosswalk["location"], crosswalk["aisle"]))

# case picks on the map
picks = pd.read_csv(PICKS_FILE, encoding="latin-1", dtype=str,
                    usecols=["NO", "CREATE_DT", "ORD_NO", "USER_NAME", "FROM_LOCATION", "FROM_UOM"])
picks = picks[(picks["FROM_UOM"] == "CS") & picks["FROM_LOCATION"].isin(stop_of)]
picks["time"] = pd.to_datetime(picks["CREATE_DT"])

# a tour = one operator's picks within one order. keep tours with 3 or more picks
picks["tour"] = picks["ORD_NO"] + " | " + picks["USER_NAME"]
picks = picks[picks.groupby("tour")["NO"].transform("size") >= 3]

def get_tours(month):
    rows = picks[picks["time"].dt.strftime("%Y-%m") == month].sort_values(["tour", "time"])
    return {tour: list(r["FROM_LOCATION"]) for tour, r in rows.groupby("tour")}   # tour -> locations in the order picked

train_tours = get_tours(TRAIN_MONTH)
test_tours = get_tours(TEST_MONTH)

In [ ]:
def tour_length(locations):
    stops = [DOCK] + [stop_of[loc] for loc in locations] + [DOCK]
    return sum(feet(stops[i], stops[i + 1]) for i in range(len(stops) - 1))

# detour = extra feet a stop adds to its tour
#        = (stop before -> this stop) + (this stop -> stop after) - (stop before -> stop after)
detour_total = {}    # location -> detour feet added up over the month
visit_count = {}     # location -> number of visits in the month
for locations in train_tours.values():
    locations = [loc for i, loc in enumerate(locations) if i == 0 or loc != locations[i - 1]]   # picks in a row at one spot = one visit
    stops = [DOCK] + [stop_of[loc] for loc in locations] + [DOCK]
    for i, loc in enumerate(locations, start=1):
        before, here, after = stops[i - 1], stops[i], stops[i + 1]
        detour = feet(before, here) + feet(here, after) - feet(before, after)
        detour_total[loc] = detour_total.get(loc, 0) + max(detour, 0)
        visit_count[loc] = visit_count.get(loc, 0) + 1

ranking = pd.DataFrame({"visits": visit_count, "detour_ft": detour_total})
ranking = ranking.sort_values("detour_ft", ascending=False)      # worst locations first
ranking.head(10)

In [ ]:
# golden zone: locations within GOLDEN_FT feet of the front of their aisle
feet_from_dock = {loc: feet(DOCK, stop) for loc, stop in stop_of.items()}
aisle_front = {}     # aisle -> feet from the dock to its closest location
for loc, aisle in aisle_of.items():
    aisle_front[aisle] = min(aisle_front.get(aisle, 1e12), feet_from_dock[loc])
golden = {loc for loc in stop_of if feet_from_dock[loc] - aisle_front[aisle_of[loc]] <= GOLDEN_FT}
print(len(golden), "golden locations")

# if a product moves, the operator still walks the tour, but the moved stop goes wherever it adds the fewest feet
def replan(locations, moved_to):
    route = [loc for loc in locations if loc not in moved_to]          # stops that did not move keep their order
    for loc in locations:
        if loc in moved_to:
            new_stop = stop_of[moved_to[loc]]
            def feet_added(position):
                before = stop_of[route[position - 1]] if position > 0 else DOCK
                after = stop_of[route[position]] if position < len(route) else DOCK
                return feet(before, new_stop) + feet(new_stop, after) - feet(before, after)
            best_position = min(range(len(route) + 1), key=feet_added)
            route.insert(best_position, moved_to[loc])
    return route

In [ ]:
tours_through = {}   # location -> training tours that stop there
for tour, locations in train_tours.items():
    for loc in set(locations):
        tours_through.setdefault(loc, []).append(tour)

quietest_golden = [loc for loc in ranking.sort_values("visits").index if loc in golden]
moved_to = {}        # location -> where its product now sits
swaps = []

for original_location in ranking.index:                    # worst location first
    if len(swaps) == MAX_SWAPS or not quietest_golden:
        break
    if original_location in golden or original_location in moved_to:
        continue
    golden_location = quietest_golden[0]                   # swap it with the quietest golden location
    tours = set(tours_through.get(original_location, [])) | set(tours_through.get(golden_location, []))
    with_swap = {**moved_to, original_location: golden_location, golden_location: original_location}
    feet_before = sum(tour_length(replan(train_tours[t], moved_to)) for t in tours)
    feet_after = sum(tour_length(replan(train_tours[t], with_swap)) for t in tours)
    if (feet_before - feet_after) * DOLLARS_PER_FOOT < 2 * COST_PER_MOVE:    # saves less than the swap costs: skip
        continue
    quietest_golden.pop(0)
    moved_to = with_swap
    swaps.append({"original location": original_location, "golden location": golden_location,
                  "feet saved (training month)": round(feet_before - feet_after)})

swaps = pd.DataFrame(swaps)
print(len(swaps), "swaps")
swaps.head(10)

In [ ]:
walked_before = sum(tour_length(locations) for locations in test_tours.values())
walked_after = sum(tour_length(replan(locations, moved_to)) for locations in test_tours.values())
feet_saved = walked_before - walked_after
dollars_saved = feet_saved * DOLLARS_PER_FOOT
swap_cost = len(swaps) * 2 * COST_PER_MOVE

print(f"{TEST_MONTH} walking:  {walked_before:,.0f} ft  ->  {walked_after:,.0f} ft with swaps")
print(f"saved {feet_saved:,.0f} ft ({100 * feet_saved / walked_before:.1f}%)")
print(f"saved ${dollars_saved:,.0f} - swap cost ${swap_cost:,.2f} = ${dollars_saved - swap_cost:,.0f}")